# Extending Elastic Agent Builder with LlamaExtract

We combine **LlamaExtract** (schema-driven PDF extraction) with **Elastic Agent Builder**
(workflow tools + search) to process complex documents like the
[World Bank Global Economic Prospects (Jan 2026)](https://www.worldbank.org/en/publication/global-economic-prospects).

Pipeline: Define schema → Create extraction agent → Upload PDF → Workflow (extract + index) → Create agent → Ask questions.

## 1. Setup

In [ ]:
%pip install llama-cloud elasticsearch dotenv pydantic -q

  Using cached llama_cloud-1.6.0-py3-none-any.whl.metadata (7.1 kB)
Using cached llama_cloud-1.6.0-py3-none-any.whl (394 kB)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Set your API keys and URLs in a `.env` file:
```bash
LLAMA_CLOUD_API_KEY="llx-..."
ELASTICSEARCH_URL="https://..."
ELASTICSEARCH_API_KEY="your-elastic-api-key"
KIBANA_URL="https://your-kibana-url"
```

In [32]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
LLAMA_CLOUD_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")

## 2. Define the Extraction Schema

Pydantic model that defines what fields to extract from the PDF.

In [3]:
from pydantic import BaseModel, Field


class EconomicIndicator(BaseModel):
    indicator: str = Field(
        description="Name of the economic indicator (e.g., 'Annual growth of per capita investment')"
    )
    emerging_markets_value: str = Field(description="Value for emerging markets (EM)")
    frontier_markets_value: str = Field(description="Value for frontier markets (FM)")
    other_developing_value: str = Field(
        description="Value for other developing economies (ODE), if available"
    )
    period: str = Field(description="Time period for this data point (e.g., '2020-24')")


class CountryExample(BaseModel):
    country: str = Field(description="Country name")
    development_approach: str = Field(
        description="The development strategy or sector focus of this country"
    )


class GlobalEconomicReport(BaseModel):
    report_title: str = Field(description="Full title of the report")
    publication_date: str = Field(description="Publication date (e.g., 'January 2026')")
    publisher: str = Field(description="Publishing organization")
    main_topic: str = Field(
        description="The primary topic or focus of the executive summary"
    )
    executive_summary: str = Field(
        description="A concise summary of the executive summary's main findings and conclusions"
    )
    frontier_markets_population_share_2025: str = Field(
        description="Frontier markets' share of global population in 2025"
    )
    frontier_markets_gdp_share_2025: str = Field(
        description="Frontier markets' share of global GDP in 2025"
    )
    frontier_markets_gdp_per_capita_vs_emerging: str = Field(
        description="How frontier markets' GDP per capita compares to emerging markets"
    )
    per_capita_investment_growth_2000s: str = Field(
        description="Average annual per capita investment growth in frontier markets during the 2000s"
    )
    per_capita_investment_growth_2020s: str = Field(
        description="Average annual per capita investment growth in frontier markets in the early 2020s"
    )
    poverty_rate_comparison: str = Field(
        description="How poverty rates in frontier markets compare to emerging markets"
    )
    sovereign_defaults_trend: str = Field(
        description="Description of the trend in sovereign default events for frontier markets"
    )
    key_vulnerabilities: list[str] = Field(
        description="List of key vulnerabilities and risks facing frontier markets"
    )
    policy_recommendations: list[str] = Field(
        description="List of policy recommendations for frontier market policy makers"
    )
    top_performing_countries: list[CountryExample] = Field(
        description="Examples of top-performing frontier markets and their development approaches"
    )
    countries_graduated_to_high_income: list[str] = Field(
        description="Countries that graduated from frontier market to emerging market status by reaching high-income level"
    )
    economic_indicators: list[EconomicIndicator] = Field(
        description="Key economic indicators extracted from charts and tables (Figures ES.A-F)"
    )


print(GlobalEconomicReport.model_json_schema())

{'$defs': {'CountryExample': {'properties': {'country': {'description': 'Country name', 'title': 'Country', 'type': 'string'}, 'development_approach': {'description': 'The development strategy or sector focus of this country', 'title': 'Development Approach', 'type': 'string'}}, 'required': ['country', 'development_approach'], 'title': 'CountryExample', 'type': 'object'}, 'EconomicIndicator': {'properties': {'indicator': {'description': "Name of the economic indicator (e.g., 'Annual growth of per capita investment')", 'title': 'Indicator', 'type': 'string'}, 'emerging_markets_value': {'description': 'Value for emerging markets (EM)', 'title': 'Emerging Markets Value', 'type': 'string'}, 'frontier_markets_value': {'description': 'Value for frontier markets (FM)', 'title': 'Frontier Markets Value', 'type': 'string'}, 'other_developing_value': {'description': 'Value for other developing economies (ODE), if available', 'title': 'Other Developing Value', 'type': 'string'}, 'period': {'descr

## 3. Create the Extraction Agent

Create a LlamaExtract agent configured with our schema. Save the `agent.id` — you'll need it for the workflow.

In [4]:
from llama_cloud import LlamaCloud

client = LlamaCloud(api_key=LLAMA_CLOUD_API_KEY)

# Create the extraction agent with our schema
agent = client.extraction.extraction_agents.create(
    name="global-economic-prospects-extractor",
    data_schema=GlobalEconomicReport.model_json_schema(),
    config={},
)
print(f"Created extraction agent: {agent.id}")

ConflictError: Error code: 409 - {'detail': "An extraction agent with the name 'global-economic-prospects-extractor' already exists in this project."}

## 4. Create the Elasticsearch Index

Create the index with mappings matching our extraction schema. The workflow will index documents here.

In [33]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

print(f"Connected: {es.info()}")

Connected: {'name': 'instance-0000000004', 'cluster_name': 'fd31ad5f77d841d69b622c17ed64b766', 'cluster_uuid': 'YxmDkf9DSwWpvcCLv98q8A', 'version': {'number': '9.3.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '17b451d8979a29e31935fe1eb901310350b30e62', 'build_date': '2026-01-29T10:05:46.708397977Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [34]:
INDEX_NAME = "economic-reports"

mappings = {
    "mappings": {
        "properties": {
            "report_title": {"type": "text"},
            "publication_date": {"type": "keyword"},
            "publisher": {"type": "keyword"},
            "main_topic": {"type": "text"},
            "executive_summary": {"type": "text"},
            "frontier_markets_population_share_2025": {"type": "keyword"},
            "frontier_markets_gdp_share_2025": {"type": "keyword"},
            "frontier_markets_gdp_per_capita_vs_emerging": {"type": "text"},
            "per_capita_investment_growth_2000s": {"type": "keyword"},
            "per_capita_investment_growth_2020s": {"type": "keyword"},
            "poverty_rate_comparison": {"type": "text"},
            "sovereign_defaults_trend": {"type": "text"},
            "key_vulnerabilities": {"type": "text"},
            "policy_recommendations": {"type": "text"},
            "top_performing_countries": {
                "type": "nested",
                "properties": {
                    "country": {"type": "keyword"},
                    "development_approach": {"type": "text"},
                },
            },
            "countries_graduated_to_high_income": {"type": "keyword"},
            "economic_indicators": {
                "type": "nested",
                "properties": {
                    "indicator": {"type": "text"},
                    "emerging_markets_value": {"type": "keyword"},
                    "frontier_markets_value": {"type": "keyword"},
                    "other_developing_value": {"type": "keyword"},
                    "period": {"type": "keyword"},
                },
            },
            "source_url": {"type": "keyword"},
        }
    }
}

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(index=INDEX_NAME, body=mappings)
print(f"Created index: {INDEX_NAME}")

Created index: economic-reports


## 5. Upload PDF to LlamaCloud

Download the PDF and upload it to LlamaCloud. The `file_id` will be used as input for the workflow.

In [8]:
import requests as req

PDF_URL = "https://raw.githubusercontent.com/Delacrobix/notebook-llamaindex-agent-builder/main/Markets-Executive-Summary.pdf"

# Download the PDF
pdf_response = req.get(PDF_URL)
pdf_response.raise_for_status()

# Upload to LlamaCloud
file_obj = client.files.create(
    file=("Markets-Executive-Summary.pdf", pdf_response.content, "application/pdf"),
    purpose="extract",
)
print(f"Uploaded file: {file_obj.id}")

Uploaded file: c1537697-ff84-4e3f-aee7-b70f9b0b784a


## 6. Workflow Tool for Elastic Agent Builder

**Manual setup required:** Copy the YAML below and paste it in the Elastic UI at
**Management → AI Assistants & Agents → Workflows → Create Workflow**. You'll need to
manually insert the input values and add `llama_cloud_api_key` under **Secrets**.

> **Production improvement:** To avoid the manual upload step, you could index the
> LlamaCloud `file_id` values into Elasticsearch alongside each document. This way,
> the workflow could look up the `file_id` directly from ES, enabling a fully automated
> pipeline without manual intervention.

```yaml
name: LlamaExtract Economic Report Processor
description: >
  Receives a file_id from LlamaCloud, runs extraction,
  and indexes the structured results into Elasticsearch.
enabled: true

inputs:
  - name: file_id
    type: string
    description: LlamaCloud file ID (from a PDF already uploaded via the SDK)
    required: true
  - name: extraction_agent_id
    type: string
    description: LlamaExtract extraction agent ID (pre-configured with schema)
    required: true
  - name: document_id
    type: string
    description: Unique identifier for the indexed document
    required: true
  - name: llama_cloud_api_key
    type: string
    description: LlamaCloud API key for authentication
    required: true

consts:
  indexName: economic-reports
  llamaExtractBaseUrl: https://api.cloud.llamaindex.ai/api/v1

triggers:
  - type: manual

steps:
  - name: create_extraction_job
    type: http
    with:
      url: "{{ consts.llamaExtractBaseUrl }}/extraction/jobs"
      method: POST
      headers:
        Authorization: "Bearer {{ inputs.llama_cloud_api_key }}"
        Content-Type: application/json
      body: |
        {
          "extraction_agent_id": "{{ inputs.extraction_agent_id }}",
          "file_id": "{{ inputs.file_id }}"
        }
      timeout: 30s

  # Wait for the extraction job to process
  - name: wait_for_extraction
    type: wait
    with:
      duration: "15s"

  # Retrieve results — retries every 10s up to 12 times (2 min total)
  - name: get_extraction_result
    type: http
    on-failure:
      retry:
        max-attempts: 12
        delay: "10s"
    with:
      url: "{{ consts.llamaExtractBaseUrl }}/extraction/jobs/{{ steps.create_extraction_job.output.body.id }}/result"
      method: GET
      headers:
        Authorization: "Bearer {{ inputs.llama_cloud_api_key }}"
        Accept: application/json
      timeout: 120s

  - name: index_extracted_data
    type: elasticsearch.index
    with:
      index: "{{ consts.indexName }}"
      id: "{{ inputs.document_id }}"
      document:
        report_title: "{{ steps.get_extraction_result.output.body.data.report_title }}"
        publication_date: "{{ steps.get_extraction_result.output.body.data.publication_date }}"
        publisher: "{{ steps.get_extraction_result.output.body.data.publisher }}"
        main_topic: "{{ steps.get_extraction_result.output.body.data.main_topic }}"
        executive_summary: "{{ steps.get_extraction_result.output.body.data.executive_summary }}"
        frontier_markets_population_share_2025: "{{ steps.get_extraction_result.output.body.data.frontier_markets_population_share_2025 }}"
        frontier_markets_gdp_share_2025: "{{ steps.get_extraction_result.output.body.data.frontier_markets_gdp_share_2025 }}"
        frontier_markets_gdp_per_capita_vs_emerging: "{{ steps.get_extraction_result.output.body.data.frontier_markets_gdp_per_capita_vs_emerging }}"
        per_capita_investment_growth_2000s: "{{ steps.get_extraction_result.output.body.data.per_capita_investment_growth_2000s }}"
        per_capita_investment_growth_2020s: "{{ steps.get_extraction_result.output.body.data.per_capita_investment_growth_2020s }}"
        poverty_rate_comparison: "{{ steps.get_extraction_result.output.body.data.poverty_rate_comparison }}"
        sovereign_defaults_trend: "{{ steps.get_extraction_result.output.body.data.sovereign_defaults_trend }}"
        key_vulnerabilities: "{{ steps.get_extraction_result.output.body.data.key_vulnerabilities | json }}"
        policy_recommendations: "{{ steps.get_extraction_result.output.body.data.policy_recommendations | json }}"
        top_performing_countries: "{{ steps.get_extraction_result.output.body.data.top_performing_countries | json }}"
        countries_graduated_to_high_income: "{{ steps.get_extraction_result.output.body.data.countries_graduated_to_high_income | json }}"
        economic_indicators: "{{ steps.get_extraction_result.output.body.data.economic_indicators | json }}"
      refresh: wait_for

  - name: verify_document
    type: elasticsearch.search
    with:
      index: "{{ consts.indexName }}"
      query:
        term:
          _id: "{{ inputs.document_id }}"
```

## 7. Create the Agent via API

Create a workflow tool that calls the extraction workflow, and an ES|QL tool to query the indexed results.
Then create an agent that uses both tools. We use the [Agent Builder Kibana APIs](https://www.elastic.co/docs/explore-analyze/ai-features/agent-builder/kibana-api).

**Note:** After creating the workflow in step 6, copy its ID from the Elastic UI and set it below as `WORKFLOW_ID`.

In [20]:
import requests

WORKFLOW_ID = "workflow-a3b8a34b-0073-4a08-98a8-63ba79953be0"  # Copy from Elastic UI after creating the workflow

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

print(WORKFLOW_ID)

workflow-a3b8a34b-0073-4a08-98a8-63ba79953be0


In [22]:
# Create a workflow tool that triggers the extraction + indexing pipeline
workflow_tool_payload = {
    "id": "run_extraction_workflow",
    "type": "workflow",
    "description": (
        "Triggers the LlamaExtract extraction workflow. "
        "Use this tool to extract structured data from an uploaded PDF and index it into Elasticsearch. "
        "Requires file_id (LlamaCloud file ID), extraction_agent_id, document_id, and llama_cloud_api_key."
    ),
    "tags": ["llama-extract", "workflow"],
    "configuration": {
        "workflow_id": WORKFLOW_ID,
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=workflow_tool_payload,
)
print(f"Workflow tool created: {response.status_code}")
print(response.json())

Workflow tool created: 200
{'id': 'run_extraction_workflow', 'type': 'workflow', 'description': 'Triggers the LlamaExtract extraction workflow. Use this tool to extract structured data from an uploaded PDF and index it into Elasticsearch. Requires file_id (LlamaCloud file ID), extraction_agent_id, document_id, and llama_cloud_api_key.', 'tags': ['llama-extract', 'workflow'], 'configuration': {'workflow_id': 'workflow-a3b8a34b-0073-4a08-98a8-63ba79953be0'}, 'readonly': False, 'schema': {'type': 'object', 'properties': {'file_id': {'type': 'string', 'description': 'LlamaCloud file ID (from a PDF already uploaded via the SDK)'}, 'extraction_agent_id': {'type': 'string', 'description': 'LlamaExtract extraction agent ID (pre-configured with schema)'}, 'document_id': {'type': 'string', 'description': 'Unique identifier for the indexed document'}, 'llama_cloud_api_key': {'type': 'string', 'description': 'LlamaCloud API key for authentication'}}, 'required': ['file_id', 'extraction_agent_id', 

In [23]:
# Create an ES|QL tool to query the indexed economic reports
query_tool_payload = {
    "id": "query_economic_reports",
    "type": "esql",
    "description": (
        "Search economic report data already indexed in Elasticsearch. "
        "Use this to answer questions about frontier markets, economic indicators, "
        "policy recommendations, and country performance."
    ),
    "tags": ["economic-data", "llama-extract"],
    "configuration": {
        "query": (
            "FROM economic-reports "
            "| WHERE MATCH(executive_summary, ?query) "
            "OR MATCH(key_vulnerabilities, ?query) "
            "OR MATCH(policy_recommendations, ?query) "
            "| LIMIT 5"
        ),
        "params": {
            "query": {
                "type": "keyword",
                "description": "The search query to find relevant economic data",
            }
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=query_tool_payload,
)
print(f"Query tool created: {response.status_code}")
print(response.json())

Query tool created: 200
{'id': 'query_economic_reports', 'type': 'esql', 'description': 'Search economic report data already indexed in Elasticsearch. Use this to answer questions about frontier markets, economic indicators, policy recommendations, and country performance.', 'tags': ['economic-data', 'llama-extract'], 'configuration': {'query': 'FROM economic-reports | WHERE MATCH(executive_summary, ?query) OR MATCH(key_vulnerabilities, ?query) OR MATCH(policy_recommendations, ?query) | LIMIT 5', 'params': {'query': {'type': 'keyword', 'description': 'The search query to find relevant economic data'}}}, 'readonly': False, 'schema': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'The search query to find relevant economic data'}}, 'required': ['query'], 'additionalProperties': False, 'description': 'Parameters needed to execute the query', '$schema': 'http://json-schema.org/draft-07/schema#'}}


In [24]:
# Create the agent with both tools
agent_payload = {
    "id": "economic-report-analyst",
    "name": "Economic Report Analyst",
    "description": "Extracts and analyzes economic reports from PDFs using LlamaExtract and Elasticsearch.",
    "labels": ["economics", "llama-extract"],
    "configuration": {
        "instructions": (
            "You are an economic research assistant. You have two tools:\n"
            "1. run_extraction_workflow: Use this FIRST to extract and index data from a PDF.\n"
            "2. query_economic_reports: Use this AFTER extraction to search the indexed data and answer questions.\n"
            "When presenting data, use clear formatting with bullet points or tables. "
            "Always cite the report title and publication date."
        ),
        "tools": [
            {
                "tool_ids": [
                    "run_extraction_workflow",
                    "query_economic_reports",
                ]
            }
        ],
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json=agent_payload,
)
print(f"Agent created: {response.status_code}")
print(response.json())

Agent created: 200
{'id': 'economic-report-analyst', 'type': 'chat', 'name': 'Economic Report Analyst', 'description': 'Extracts and analyzes economic reports from PDFs using LlamaExtract and Elasticsearch.', 'labels': ['economics', 'llama-extract'], 'configuration': {'instructions': 'You are an economic research assistant. You have two tools:\n1. run_extraction_workflow: Use this FIRST to extract and index data from a PDF.\n2. query_economic_reports: Use this AFTER extraction to search the indexed data and answer questions.\nWhen presenting data, use clear formatting with bullet points or tables. Always cite the report title and publication date.', 'tools': [{'tool_ids': ['run_extraction_workflow', 'query_economic_reports']}]}, 'readonly': False}


```json
{
    "file_id": "c1537697-ff84-4e3f-aee7-b70f9b0b784a",
    "extraction_agent_id": "e1f9fa9a-7bf0-4a2b-8d4e-208b0ebf7674",
    "document_id": "world-bank-gep-jan-2026",
    "llama_cloud_api_key": "llx-tu-api-key-aqui",
}
```

In [ ]:
extraction_agent_id = (
    "e1f9fa9a-7bf0-4a2b-8d4e-208b0ebf7674"  # Copy from response when creating the agent
)


# Ask the agent to extract and analyze the report
converse_payload = {
    "agent_id": "economic-report-analyst",
    "input": (
        f"Extract the economic report from this PDF (file_id: {file_obj.id}, "
        f"extraction_agent_id: {extraction_agent_id}, document_id: global-economic-prospects-jan-2026, "
        f"llama_cloud_api_key: {LLAMA_CLOUD_API_KEY}). "
        "Then tell me: what are the main vulnerabilities facing frontier markets?"
    ),
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=converse_payload,
)
print(json.dumps(response.json(), indent=2))

{
  "conversation_id": "2d32b5ec-5955-4791-ae0a-5ca33d77dd95",
  "steps": [
    {
      "type": "reasoning",
      "reasoning": "The user has provided a PDF file containing an economic report and wants information extracted from it. I need to first run the extraction workflow to index the document into Elasticsearch before I can search for specific information about frontier markets."
    },
    {
      "type": "tool_call",
      "tool_call_id": "tooluse_6FIdlPdas1g7BxQAxi5wYy",
      "tool_id": "run_extraction_workflow",
      "progression": [],
      "params": {
        "file_id": "c1537697-ff84-4e3f-aee7-b70f9b0b784a",
        "extraction_agent_id": "e1f9fa9a-7bf0-4a2b-8d4e-208b0ebf7674",
        "document_id": "global-economic-prospects-jan-2026",
        "llama_cloud_api_key": "llx-C3ZymDMSMbtdVH1P0l17XtnV0xNHFFmMci8Gadi7GXWuVlxZ"
      },
      "results": [
        {
          "type": "other",
          "data": {
            "execution": {
              "execution_id": "9a9e1d01-

## Cleanup

In [31]:
es.indices.delete(index=INDEX_NAME)

ObjectApiResponse({'acknowledged': True})